In [ ]:
!python run.py 

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim import lr_scheduler
import torch.backends.cudnn as cudnn
import numpy as np
import torchvision
from torchvision import datasets, models, transforms
import matplotlib.pyplot as plt
import time
import os
import copy
from PIL import Image
from models.resnet_simclr import ResNetSimCLR
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [4]:
model = ResNetSimCLR("resnet18",32).cuda()
print(model)
state_dict = torch.load('simclr_resnet.pkl', map_location=device)
model.load_state_dict(state_dict, strict=False)

ResNetSimCLR(
  (backbone): ResNet(
    (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (relu): ReLU(inplace=True)
    (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (layer1): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, tra

<All keys matched successfully>

In [ ]:
print(model)

In [ ]:
transform_test = transforms.Compose([transforms.PILToTensor()])
transform_normalize = transforms.Compose([transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])])

In [ ]:
from pytorch_grad_cam import GradCAMPlusPlus
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image, preprocess_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
import numpy as np
from PIL import Image
from skimage.transform import resize

In [ ]:
def crop(img):
    img = np.array(img)
    w, h, c = img.shape
    if(h!=1000):
        img = resize(img[:,:,0],(800,1000))*255
        img_temp = img[:600,:]
        img_final = np.zeros((600,1000,3))
        img_final[:,:,0] = img_temp
        img_final[:,:,1] = img_temp
        img_final[:,:,2] = img_temp
    else:
        img_final = img[:600]
    img = Image.fromarray(np.uint8(img_final))
    return img

In [ ]:
model.eval()
PATH = "/home/r10941036/Explainable AI Project/DATA/val/Psoriasis/"
NAME = "20220323_170008B.png"
target_layers = [model.backbone.trunk_output.block4[1].f.c[0]]
cam = GradCAMPlusPlus(model=model, target_layers=target_layers, use_cuda=True)
targets = [ClassifierOutputTarget(0)]
img = Image.open(PATH+NAME).convert('RGB')
test = transform_normalize(transform_test(img)/255).unsqueeze(0)
test = test.to(device)
result = model(test)
sfm = torch.nn.Softmax()
rgb_img = np.float32(img)/255
input_tensor = preprocess_image(rgb_img)
grayscale_cam2 = cam(input_tensor=input_tensor,targets=targets)[0,:]
visualization = show_cam_on_image(rgb_img, grayscale_cam2, use_rgb=True)

In [ ]:
plt.rcParams["figure.figsize"] = (20,20)
plt.imshow(rgb_img,cmap="gray")
plt.axis("off")
plt.show()
plt.rcParams["figure.figsize"] = (20,20)
plt.imshow(visualization,cmap="gray")
plt.axis("off")
plt.show()